In [1]:
!pip install -q torch transformers scikit-learn tqdm sentencepiece huggingface_hub accelerate


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
!pip install -q torch transformers scikit-learn tqdm sentencepiece huggingface_hub accelerate protobuf


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
from huggingface_hub import login
login("")

In [6]:
import os
import json
import random
import importlib.util

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from tqdm import tqdm

SEED = 42
MODEL_NAME = "microsoft/deberta-v3-base"
HF_TOKEN = os.getenv("HF_TOKEN", "your_hf_token_here")

TRAIN_PATH = "train_data/subtask 1/train_data.json"
TEST_PATH = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"

OUT_DIR = "deberta_subtask1_ft"

TRAIN_SPLIT_PATH = os.path.join(OUT_DIR, "train_split.json")
DEV_SPLIT_PATH = os.path.join(OUT_DIR, "dev_split.json")
DEV_PRED_PATH = os.path.join(OUT_DIR, "dev_predictions.json")
DEV_EVAL_PATH = os.path.join(OUT_DIR, "dev_official_eval.json")
TEST_PRED_PATH = os.path.join(OUT_DIR, "test_predictions.json")
BEST_MODEL_PATH = os.path.join(OUT_DIR, "best_model.pt")

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01
DROPOUT = 0.1
WARMUP_RATIO = 0.1
DEV_RATIO = 0.2

PADDING_MODE = "dynamic"  # "dynamic" or "static"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj, path):
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def stratified_split(items, dev_ratio, seed):
    rng = random.Random(seed)
    buckets = {}
    for it in items:
        key = (bool(it["validity"]), bool(it["plausibility"]))
        buckets.setdefault(key, []).append(it)

    train, dev = [], []
    for bucket in buckets.values():
        rng.shuffle(bucket)
        if len(bucket) <= 1:
            n_dev = 0
        else:
            n_dev = max(1, int(round(len(bucket) * dev_ratio)))
            n_dev = min(n_dev, len(bucket) - 1)
        dev.extend(bucket[:n_dev])
        train.extend(bucket[n_dev:])

    rng.shuffle(train)
    rng.shuffle(dev)
    return train, dev

class SyllogismDataset(Dataset):
    def __init__(self, data, tokenizer, max_length, padding_mode="dynamic", is_test=False):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.padding_mode = padding_mode
        self.is_test = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        if self.padding_mode == "static":
            enc = self.tokenizer(
                item["syllogism"],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
            )
        else:
            enc = self.tokenizer(
                item["syllogism"],
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )

        out = {
            "id": item["id"],
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }

        if not self.is_test:
            out["labels"] = torch.tensor(int(item["validity"]), dtype=torch.long)

        return out

def static_collate_fn(batch):
    result = {
        "id": [x["id"] for x in batch],
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
    }
    if "labels" in batch[0]:
        result["labels"] = torch.stack([x["labels"] for x in batch])
    return result

def make_dynamic_collator(tokenizer):
    padder = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt")

    def collate_fn(batch):
        ids = [x["id"] for x in batch]
        features = []
        for x in batch:
            feat = {
                "input_ids": x["input_ids"],
                "attention_mask": x["attention_mask"],
            }
            if "labels" in x:
                feat["labels"] = x["labels"]
            features.append(feat)
        padded = padder(features)
        padded["id"] = ids
        return padded

    return collate_fn

def evaluate_loss_acc(model, loader, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    acc = accuracy_score(all_labels, all_preds) * 100.0
    return total_loss / max(len(loader), 1), acc

@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    preds_out = []

    for batch in loader:
        ids = batch["id"]
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().tolist()

        for ex_id, pred in zip(ids, preds):
            preds_out.append({"id": ex_id, "validity": bool(pred)})

    return preds_out

def run_official_eval(eval_script_path, ref_path, pred_path, out_path):
    spec = importlib.util.spec_from_file_location("semeval_eval", eval_script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.run_full_scoring(ref_path, pred_path, out_path)
    return load_json(out_path)

def main():
    if PADDING_MODE not in {"dynamic", "static"}:
        raise ValueError("PADDING_MODE must be 'dynamic' or 'static'")

    set_seed(SEED)
    os.makedirs(OUT_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_data = load_json(TRAIN_PATH)
    test_data = load_json(TEST_PATH)

    train_split, dev_split = stratified_split(train_data, DEV_RATIO, SEED)
    save_json(train_split, TRAIN_SPLIT_PATH)
    save_json(dev_split, DEV_SPLIT_PATH)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
        token=HF_TOKEN,
    ).to(device)

    train_ds = SyllogismDataset(
        train_split,
        tokenizer,
        MAX_LENGTH,
        padding_mode=PADDING_MODE,
        is_test=False,
    )
    dev_ds = SyllogismDataset(
        dev_split,
        tokenizer,
        MAX_LENGTH,
        padding_mode=PADDING_MODE,
        is_test=False,
    )
    test_ds = SyllogismDataset(
        test_data,
        tokenizer,
        MAX_LENGTH,
        padding_mode=PADDING_MODE,
        is_test=True,
    )

    if PADDING_MODE == "dynamic":
        collate_fn = make_dynamic_collator(tokenizer)
    else:
        collate_fn = static_collate_fn

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_score = -1.0

    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0.0
        loop = tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}")

        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_train_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        train_loss = total_train_loss / max(len(train_loader), 1)
        dev_loss, dev_acc = evaluate_loss_acc(model, dev_loader, device)

        dev_preds = predict(model, dev_loader, device)
        save_json(dev_preds, DEV_PRED_PATH)
        dev_metrics = run_official_eval(EVAL_SCRIPT, DEV_SPLIT_PATH, DEV_PRED_PATH, DEV_EVAL_PATH)
        dev_combined = dev_metrics["combined_score"]

        print(
            json.dumps(
                {
                    "epoch": epoch + 1,
                    "padding_mode": PADDING_MODE,
                    "train_loss": round(train_loss, 4),
                    "dev_loss": round(dev_loss, 4),
                    "dev_acc": round(dev_acc, 4),
                    "dev_content_effect": round(dev_metrics["content_effect"], 4),
                    "dev_combined_score": round(dev_combined, 4),
                    "best_combined_score": round(max(best_score, dev_combined), 4),
                }
            )
        )

        if dev_combined > best_score:
            best_score = dev_combined
            torch.save(model.state_dict(), BEST_MODEL_PATH)

    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    test_preds = predict(model, test_loader, device)
    save_json(test_preds, TEST_PRED_PATH)
    print(f"saved test predictions to {TEST_PRED_PATH}")

if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]